# PRACTICA OBLIGATORIA: Transformación y Limpieza
Dataset: Netflix Titles

In [ ]:
import numpy as np
import pandas as pd

df_peliculas = pd.read_csv("data/dataset_netflix_titles.csv")

## #0 Carga de datos y primera exploración

### Ejercicio 1: Primera exploración del dataset

In [ ]:
# Visualización general de las primeras filas
# .head() muestra las 5 primeras filas por defecto, útil para ver cómo luce el dato
print("=== Primeras filas del dataset ===")
df_peliculas.head()

In [ ]:
# Información general: tipos de datos, cantidad de no-nulos por columna y uso de memoria
# .info() es clave en la exploración inicial porque detecta rápidamente columnas con nulos
# y si los tipos inferidos por pandas son correctos (ej: fechas como object, etc.)
print("=== Información general del dataset ===")
df_peliculas.info()

In [ ]:
# Descripción estadística de las columnas numéricas
# .describe() calcula count, mean, std, min, percentiles y max
# Nos permite ver si hay valores anómalos (ej: release_year con valores imposibles)
print("=== Descripción de variables numéricas ===")
df_peliculas.describe()

In [ ]:
# Nombre de todas las columnas del DataFrame
print("=== Columnas del dataset ===")
print(df_peliculas.columns.tolist())

In [ ]:
# Distribución de 3 columnas escogidas: 'type', 'rating' y 'release_year'
# .value_counts() cuenta las ocurrencias de cada valor único en una columna categórica
# Es la forma más directa de entender cómo se distribuyen los datos

print("=== Distribución de 'type' ===")
print(df_peliculas['type'].value_counts())

print("\n=== Distribución de 'rating' ===")
print(df_peliculas['rating'].value_counts())

print("\n=== Distribución de 'release_year' (top 10) ===")
print(df_peliculas['release_year'].value_counts().head(10))

## #1 Duplicados y cardinalidad

### Ejercicio 1: ¿Existen filas duplicadas? ¿Cuántas?

In [ ]:
# .duplicated() devuelve una Serie booleana: True si la fila es duplicada (ya apareció antes)
# .sum() cuenta los True (duplicados), ya que True == 1 en Python
num_duplicados = df_peliculas.duplicated().sum()
print(f"Número de filas duplicadas: {num_duplicados}")

### Ejercicio 2: Eliminar duplicados, quedarse con las últimas copias

In [ ]:
# .drop_duplicates(keep='last') elimina duplicados conservando la última ocurrencia
# keep='last' vs keep='first': first guarda la primera aparición, last guarda la última
# Esto es relevante cuando los datos más recientes suelen estar al final del dataset
# reset_index(drop=True) reinicia el índice tras eliminar filas para que sea continuo
df_peliculas = df_peliculas.drop_duplicates(keep='last').reset_index(drop=True)
print(f"Shape tras eliminar duplicados: {df_peliculas.shape}")

### Ejercicio 3: Calcular la cardinalidad de cada columna (con diccionario)

In [ ]:
# La cardinalidad es el número de valores únicos de una columna en relación al total de filas
# Expresada en %, nos dice qué tan "variada" es una columna
# 100% = identificador único, 0% aprox = casi constante
# nunique() cuenta valores únicos EXCLUYENDO NaN por defecto
# Para incluir NaN en la cardinalidad usamos nunique(dropna=False)

total_filas = len(df_peliculas)

cardinalidad = {}
for col in df_peliculas.columns:
    # nunique(dropna=False) incluye NaN como un valor único más
    n_unicos = df_peliculas[col].nunique(dropna=False)
    cardinalidad[col] = round((n_unicos / total_filas) * 100, 2)

print("=== Cardinalidad por columna (%) ===")
for col, card in cardinalidad.items():
    print(f"  {col}: {card}%")

### Ejercicio 4: Detectar columnas que puedan ser un buen índice (cardinalidad = 100%)

In [ ]:
# Una columna es buen índice si cada fila tiene un valor completamente único (100%)
# Esto implica que podría usarse como clave primaria en una base de datos
print("=== Columnas candidatas a índice (cardinalidad = 100%) ===")
buenos_indices = [col for col, card in cardinalidad.items() if card == 100.0]

if buenos_indices:
    for col in buenos_indices:
        print(f"  → '{col}' puede ser un buen índice")
else:
    print("  No hay columnas con cardinalidad del 100%")

### BONUS: Ejercicios 3 y 4 usando un objeto Series

In [ ]:
# pd.Series permite construir el diccionario de cardinalidad como una estructura nativa de pandas
# Ventaja: podemos usar .apply(), filtros vectorizados y métodos de Series directamente
# sin necesidad de iterar manualmente con un bucle for

# Calculamos la cardinalidad usando apply sobre las columnas del DataFrame
cardinalidad_series = df_peliculas.apply(
    lambda col: round((col.nunique(dropna=False) / total_filas) * 100, 2)
)
# El resultado es una Series con nombre de columna como índice y cardinalidad como valor

print("=== Cardinalidad (Series) ===")
print(cardinalidad_series)

print("\n=== Buenos índices (cardinalidad = 100%) ===")
# Filtrado booleano sobre la Series: más limpio que iterar un diccionario
buenos_indices_series = cardinalidad_series[cardinalidad_series == 100.0]
if len(buenos_indices_series) > 0:
    print(buenos_indices_series)
else:
    print("No hay columnas con cardinalidad del 100%")

## #2 Limpieza y transformación (I)

### Ejercicio 1: Quitar espacios en blanco al inicio y final de los valores string

In [ ]:
# .str.strip() elimina espacios al inicio y al final de strings
# Los espacios ocultos son una fuente clásica de valores aparentemente duplicados que no se detectan
# Ejemplo: 'United States' != ' United States' para Python, pero visualmente son iguales
# Iteramos columna a columna y solo aplicamos strip a las que son de tipo object (texto)

for col in df_peliculas.columns:
    if df_peliculas[col].dtype == 'object':
        df_peliculas[col] = df_peliculas[col].str.strip()

print("Strip aplicado a todas las columnas de tipo string.")

In [ ]:
# BONUS: Forma rápida de verificar si el strip tuvo efecto en la cardinalidad
# Comparamos la cardinalidad antes y después usando la Series recalculada
cardinalidad_post_strip = df_peliculas.apply(
    lambda col: round((col.nunique(dropna=False) / total_filas) * 100, 2)
)

# Mostramos solo columnas donde hubo cambio
diff = cardinalidad_series.compare(cardinalidad_post_strip)
if diff.empty:
    print("El strip no modificó la cardinalidad de ninguna columna.")
else:
    print("Columnas donde cambió la cardinalidad tras el strip:")
    print(diff)

### Ejercicio 2: Listar columnas con cardinalidad menor al 10%

In [ ]:
# Recalculamos con los datos ya limpios (sin espacios)
# Columnas con baja cardinalidad (<10%) son columnas categóricas o semi-categóricas
# Son las más interesantes para limpieza porque sus pocos valores únicos son auditables manualmente
cardinalidad_series = df_peliculas.apply(
    lambda col: round((col.nunique(dropna=False) / total_filas) * 100, 2)
)

cols_baja_cardinalidad = cardinalidad_series[cardinalidad_series < 10].index.tolist()
print("=== Columnas con cardinalidad < 10% ===")
for col in cols_baja_cardinalidad:
    print(f"  {col}: {cardinalidad_series[col]}%")

### Ejercicio 3: Distribución de valores, valores únicos y análisis de limpieza necesaria

In [ ]:
# Para cada columna de baja cardinalidad, mostramos:
# - value_counts(): distribución de frecuencias (cuántas veces aparece cada valor)
# - unique(): lista de valores únicos (para detectar valores extraños a simple vista)
# Esto nos permite decidir si hay valores sucios antes de actuar

for col in cols_baja_cardinalidad:
    print(f"\n{'='*50}")
    print(f"Columna: '{col}'")
    print(f"Valores únicos: {df_peliculas[col].nunique(dropna=False)}")
    print(f"\nDistribución de valores:")
    print(df_peliculas[col].value_counts(dropna=False))
    print(f"\nValores únicos: {df_peliculas[col].unique()}")

### Ejercicio 4: Limpiar/transformar dos de los campos detectados

In [ ]:
# --- Campo 1: 'rating' ---
# Observamos que 'rating' puede contener valores que en realidad son duraciones
# (ej: '74 min', '84 min') que han sido mal colocados en esa columna
# Los valores válidos de rating son: TV-MA, TV-14, TV-PG, R, PG-13, etc.
# Estrategia: convertir los valores inválidos a NaN y luego imputar con la moda

# Definimos los ratings válidos conocidos en Netflix
ratings_validos = ['TV-MA', 'TV-14', 'TV-PG', 'TV-G', 'TV-Y', 'TV-Y7', 'TV-Y7-FV',
                   'R', 'PG-13', 'PG', 'G', 'NC-17', 'NR', 'UR']

# Detectamos cuántos valores inválidos hay
invalidos_rating = ~df_peliculas['rating'].isin(ratings_validos) & df_peliculas['rating'].notna()
print(f"Valores inválidos en 'rating': {invalidos_rating.sum()}")
print(df_peliculas.loc[invalidos_rating, 'rating'].value_counts())

# Los convertimos a NaN para tratarlos en el apartado de nulos
df_peliculas.loc[invalidos_rating, 'rating'] = np.nan
print(f"\nDistribución de 'rating' tras limpieza:")
print(df_peliculas['rating'].value_counts(dropna=False))

In [ ]:
# --- Campo 2: 'type' ---
# 'type' solo debería tener 'Movie' y 'TV Show'
# Comprobamos si hay valores extraños y los convertimos a '' para luego imputar

tipos_validos = ['Movie', 'TV Show']
invalidos_type = ~df_peliculas['type'].isin(tipos_validos) & df_peliculas['type'].notna()
print(f"Valores inválidos en 'type': {invalidos_type.sum()}")

if invalidos_type.sum() > 0:
    df_peliculas.loc[invalidos_type, 'type'] = np.nan
    print("Valores extraños convertidos a NaN.")
else:
    print("'type' está limpio. No hay valores a corregir.")

print(f"\nDistribución de 'type':")
print(df_peliculas['type'].value_counts(dropna=False))

### Ejercicio 5: Convertir la columna 'date_added' a tipo datetime

In [ ]:
# pd.to_datetime() convierte strings a objetos datetime de pandas (Timestamp)
# format='%B %d, %Y' indica el patrón exacto: mes completo en inglés, día, año con 4 dígitos
# errors='coerce' convierte a NaT (Not a Time) los valores que no puedan parsearse
# en lugar de lanzar una excepción, lo que es más robusto para datos sucios
# 'release_year' se deja como integer ya que operar con años como números es más sencillo

df_peliculas['date_added'] = pd.to_datetime(
    df_peliculas['date_added'],
    format='%B %d, %Y',
    errors='coerce'
)

print(f"Tipo de 'date_added': {df_peliculas['date_added'].dtype}")
print(f"Tipo de 'release_year': {df_peliculas['release_year'].dtype}")
print(f"\nEjemplos de date_added convertido:")
print(df_peliculas['date_added'].head(10))

## #3 Tratamiento de Missings/Nulos

### Ejercicio 1: Lista de columnas con valores nulos

In [ ]:
# .isnull().sum() cuenta los NaN por columna
# Filtramos solo las que tienen al menos 1 nulo
# Esto es más limpio que leer el output de .info() manualmente

nulos_por_columna = df_peliculas.isnull().sum()
columnas_con_nulos = nulos_por_columna[nulos_por_columna > 0]

print("=== Columnas con valores nulos ===")
print(columnas_con_nulos)

### Ejercicio 2: Porcentaje de nulos por columna

In [ ]:
# Dividimos el conteo de nulos entre el total de filas y multiplicamos por 100
# round(2) para legibilidad
# Esto nos permite priorizar qué columnas tienen un problema de calidad severo

porcentaje_nulos = (df_peliculas.isnull().sum() / total_filas * 100).round(2)
porcentaje_nulos_filtrado = porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)

print("=== Porcentaje de nulos por columna ===")
print(porcentaje_nulos_filtrado)

### Ejercicio 3: Corregir las dos columnas con MENOR porcentaje de nulos

In [ ]:
# Las columnas con menos nulos son las más fáciles de imputar sin distorsionar el dataset
# .nsmallest(2) devuelve las 2 columnas con menor porcentaje (excluyendo 0)

dos_menores = porcentaje_nulos_filtrado.nsmallest(2)
print(f"Las dos columnas con menor porcentaje de nulos:")
print(dos_menores)

col1, col2 = dos_menores.index[0], dos_menores.index[1]

# Para columnas categóricas (object/string), imputamos con la MODA (valor más frecuente)
# La moda es la estrategia más neutral: no inventa valores nuevos, usa el más probable
for col in [col1, col2]:
    dtype = df_peliculas[col].dtype
    if dtype == 'object' or str(dtype) == 'category':
        moda = df_peliculas[col].mode()[0]
        df_peliculas[col] = df_peliculas[col].fillna(moda)
        print(f"'{col}' (categórica): imputada con moda → '{moda}'")
    elif np.issubdtype(dtype, np.number):
        # Para numéricas, usamos la mediana (más robusta a outliers que la media)
        mediana = df_peliculas[col].median()
        df_peliculas[col] = df_peliculas[col].fillna(mediana)
        print(f"'{col}' (numérica): imputada con mediana → {mediana}")
    else:
        # Para datetime, imputamos con la mediana temporal (método más lógico)
        mediana_dt = df_peliculas[col].dropna().sort_values().iloc[len(df_peliculas[col].dropna())//2]
        df_peliculas[col] = df_peliculas[col].fillna(mediana_dt)
        print(f"'{col}' (datetime): imputada con mediana temporal → {mediana_dt}")

### Ejercicio 4: Tratar los nulos restantes con 'UNK' o ''

In [ ]:
# Para las columnas que quedan con nulos (no numéricas, no categóricas clave),
# el enunciado indica usar 'UNK' o '' para marcarlos explícitamente
# Esto es preferible a dejar NaN porque:
# 1. Las operaciones de string con NaN fallan
# 2. 'UNK' es un valor legible que indica "desconocido" de forma explícita

columnas_restantes_con_nulos = df_peliculas.columns[df_peliculas.isnull().any()].tolist()
print(f"Columnas con nulos restantes: {columnas_restantes_con_nulos}")

for col in columnas_restantes_con_nulos:
    dtype = df_peliculas[col].dtype
    if dtype == 'object':
        df_peliculas[col] = df_peliculas[col].fillna('UNK')
        print(f"'{col}': nulos → 'UNK'")
    else:
        # Para datetime u otros, convertimos a string primero y luego rellenamos
        df_peliculas[col] = df_peliculas[col].astype(str).replace('NaT', 'UNK')
        print(f"'{col}': NaT → 'UNK' (convertida a string)")

print(f"\nNulos tras tratamiento: {df_peliculas.isnull().sum().sum()}")

### Ejercicio 5: Comprobar que no se han generado duplicados

In [ ]:
# Al rellenar nulos con valores constantes (como 'UNK'), podríamos haber hecho
# que filas que antes eran distintas (por sus NaN) ahora sean idénticas
# Por eso es importante verificar y, si los hay, quedarse con la primera ocurrencia

duplicados_post = df_peliculas.duplicated().sum()
print(f"Duplicados tras tratamiento de nulos: {duplicados_post}")

if duplicados_post > 0:
    df_peliculas = df_peliculas.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"Duplicados eliminados. Shape final: {df_peliculas.shape}")
else:
    print("No se han generado duplicados. Dataset limpio.")

## #4 Generación de nuevos datos

### Ejercicio 1: Separar el dataset en películas y series

In [ ]:
# Filtramos por la columna 'type' que distingue Movie de TV Show
# reset_index(drop=True) es buena práctica: los índices quedan limpios y continuos
# Separar permite aplicar transformaciones específicas sin afectar al otro grupo

df_movies = df_peliculas[df_peliculas['type'] == 'Movie'].reset_index(drop=True)
df_shows  = df_peliculas[df_peliculas['type'] == 'TV Show'].reset_index(drop=True)

print(f"Películas: {len(df_movies)} registros")
print(f"Series:    {len(df_shows)} registros")

### Ejercicio 2: Distribución del campo 'duration' en cada DataFrame

In [ ]:
# En películas 'duration' es en minutos (ej: '90 min')
# En series 'duration' es en temporadas (ej: '2 Seasons')
# Ambas distribuciones tienen lógicas completamente distintas, por eso conviene tratarlas por separado

print("=== Distribución de 'duration' en PELÍCULAS ===")
print(df_movies['duration'].value_counts().head(20))

print("\n=== Distribución de 'duration' en SERIES ===")
print(df_shows['duration'].value_counts())

### Ejercicio 3: Convertir 'duration' a valores numéricos

In [ ]:
# Usamos expresiones regulares (regex) para extraer el número de cada string
# r'(\d+)' captura uno o más dígitos consecutivos
# .str.extract() aplica el patrón y devuelve el grupo capturado como string
# .astype(float) convierte a número (float para manejar posibles NaN)

# Películas: extraemos los minutos (ej: '90 min' → 90)
df_movies['duration_mins'] = (
    df_movies['duration']
    .str.extract(r'(\d+)')[0]
    .astype(float)
)

# Series: extraemos el número de temporadas (ej: '2 Seasons' → 2)
# y renombramos la columna para que sea más representativa
df_shows['duration_seasons'] = (
    df_shows['duration']
    .str.extract(r'(\d+)')[0]
    .astype(float)
)

print("=== Películas - duration_mins ===")
print(df_movies['duration_mins'].describe())

print("\n=== Series - duration_seasons ===")
print(df_shows['duration_seasons'].describe())

### BONUS: Campo 'Posible_Secuela' para el dataset de películas

In [ ]:
import re

# Lógica: una secuela suele indicarse de varias formas en el título:
# - Un número romano al final: 'Rocky II', 'Terminator 2'
# - Un número arábigo al final: 'Saw 3', 'Fast 9'
# - Palabras clave: 'Part 2', 'Chapter 2', 'Vol. 2'
# - Sufijos típicos: 'Returns', 'Rises', 'Reloaded', 'Begins'
# No es infalible, pero cubre los casos más habituales

PATRON_SECUELA = re.compile(
    r'\b(II|III|IV|VI|VII|VIII|IX|XI|XII)\b'   # Números romanos (del 2 en adelante)
    r'|\b[2-9]\b'                               # Número arábigo del 2 al 9
    r'|\bPart\s+[2-9\b]'                        # 'Part 2', 'Part 3'...
    r'|\bChapter\s+[2-9]'                       # 'Chapter 2'...
    r'|\bVol\.?\s*[2-9]'                        # 'Vol. 2', 'Vol 3'...
    r'|\b(Returns|Rises|Reloaded|Revolutions|Resurrection|Strikes Back|Forever|Beyond)\b',
    re.IGNORECASE
)

def es_posible_secuela(titulo):
    """Retorna True si el título tiene patrones típicos de secuela."""
    if pd.isna(titulo) or titulo == 'UNK':
        return False
    return bool(PATRON_SECUELA.search(titulo))

# Creamos la columna con valor False por defecto y aplicamos la función
# .apply() aplica la función fila a fila sobre la columna 'title'
df_movies['Posible_Secuela'] = False
df_movies['Posible_Secuela'] = df_movies['title'].apply(es_posible_secuela)

print(f"Películas marcadas como posible secuela: {df_movies['Posible_Secuela'].sum()}")
print("\nEjemplos detectados:")
print(df_movies[df_movies['Posible_Secuela'] == True]['title'].head(20).tolist())

In [ ]:
# Resumen final del estado del dataset
print("=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
print(f"Dataset completo: {df_peliculas.shape}")
print(f"DataFrame películas: {df_movies.shape}")
print(f"DataFrame series: {df_shows.shape}")
print(f"Nulos totales en df_peliculas: {df_peliculas.isnull().sum().sum()}")
print(f"Duplicados en df_peliculas: {df_peliculas.duplicated().sum()}")